# Clustering Applications in Finance & Banking

---

This notebook demonstrates **three practical applications** of unsupervised learning and clustering in Digital Finance & Banking. We progress from fundamentals to two real-world business cases.

## Contents

### 🎯 Part 1: Clustering Fundamentals (30 min)
Introduction to clustering concepts using synthetic data
- K-Means algorithm and implementation
- Evaluation metrics: Silhouette Score and WCSS
- Elbow method for choosing K
- Comparing K-Means vs DBSCAN

### 💳 Part 2: Credit Risk Assessment (60 min)
Clustering SME borrowers for improved default prediction
- Analyzing financial ratios of 4,500+ companies  
- Clustering as a pre-modeling step
- Comparing global vs cluster-specific credit models
- Business impact analysis

### 💰 Part 3: Company Valuation (45 min)
Using clustering to find comparable companies
- Peer company identification via clustering
- Calculating valuation multiples (EV/EBITDA)
- Estimating enterprise value for private companies
- Step-by-step valuation workflow

---

## Learning Objectives

By completing this notebook, you will:

1. ✅ Understand when and why to use clustering in finance
2. ✅ Apply K-Means and DBSCAN to financial datasets
3. ✅ Evaluate clustering quality systematically
4. ✅ Implement clustering-enhanced credit risk models
5. ✅ Execute multiples-based company valuation
6. ✅ Recognize clustering as a valuable pre-modeling technique

---

## Why Clustering in Finance?

Clustering enables **pattern discovery** in unlabeled data:

**Customer Intelligence:**
- Segment clients by behavior, profitability, risk
- Personalize products and pricing
- Identify high-value segments

**Risk Management:**
- Group borrowers with similar default risk
- Detect fraud patterns (outlier detection)
- Model market regimes

**Portfolio Construction:**
- Cluster assets by return characteristics
- Build diversified multi-asset portfolios
- Identify investment themes

**Valuation & M&A:**
- Find comparable companies objectively
- Benchmark performance against peers
- Identify acquisition targets

---

Let's begin! 🚀

## Setup: Import Libraries

We'll use standard Python data science libraries:
- **Pandas & NumPy**: Data manipulation
- **Matplotlib & Seaborn**: Visualization
- **Scikit-learn**: Clustering algorithms and ML tools
- **SciPy & Statsmodels**: Statistical analysis

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from pylab import MaxNLocator # PyLab is a procedural interface to the Matplotlib object-oriented plotting library.
import seaborn as sns
from sklearn.preprocessing import StandardScaler
%matplotlib inline

---

# Part 1: Clustering Fundamentals

## Overview

We start with **synthetic data** to build intuition about clustering before applying it to real financial problems.

### What You'll Learn:
1. How K-Means clustering works
2. Choosing the optimal number of clusters (K)
3. Evaluating clustering quality
4. When K-Means fails (and alternatives)

### Key Concepts:

**K-Means Algorithm:**
- Partition-based clustering method
- Minimizes within-cluster variance (WCSS)
- Requires specifying K (number of clusters) upfront

**Evaluation Metrics:**
- **Silhouette Score** (range: -1 to +1): Measures cluster separation
  - +1: Perfect, well-separated clusters
  - 0: Overlapping cluster boundaries
  - -1: Points likely in wrong clusters
  
- **WCSS (Within-Cluster Sum of Squares)**: Total variance within clusters
  - Lower values indicate tighter, more compact clusters
  - Use Elbow Method to find optimal K

**DBSCAN (Density-Based):**
- Finds arbitrary-shaped clusters
- Automatically identifies outliers
- No need to specify K in advance
- Best for non-linear cluster shapes

---

In [ ]:
# To start, we create a dataset.
from sklearn.datasets import make_blobs

# create blobs
X, y = make_blobs(n_samples=200, n_features=2, centers=4, cluster_std=1.6, random_state=50)

print(X)
print(y)

In [ ]:
# Let's plot data
plt.figure(figsize=(20,10))
plt.scatter(X[:,0], X[:,1],cmap='Accent', s=70)
plt.show()

In [ ]:
# Apply k-Means
from sklearn.cluster import KMeans
from sklearn import metrics
# from sklearn.metrics import pairwise_distances


# silhouette: 1=good, 0=overlap, -1=bad
# Within Cluster Sum of Squares: lower is better

def cluster_kmeans(df, nclust):

    kmeans = KMeans(n_clusters=nclust, random_state=0).fit(df)
    label = kmeans.labels_
    centroids = kmeans.cluster_centers_
    sil=metrics.silhouette_score(df, label, metric='euclidean', random_state=0)
    wcss = kmeans.inertia_

    return sil, wcss, label, centroids

cluster_kmeans(X, 4)

In [ ]:
# Let's plot the clustering
sil, wcss, label, centroid = cluster_kmeans(X, 4)
plt.figure(figsize=(10,10))
plt.scatter(X[:,0], X[:,1], c=label, cmap='Accent', s=40)
plt.show()

In [ ]:
# We need to validate the number of clusters. So let's check how the WCSS and the
# Silhouette coefficient change if we consider different number of clusters

max_n_clusters = 7

tab=pd.DataFrame(columns = ['Clusters', 'Silhouette(max)', 'WCSS(min)'], dtype=int).fillna('')
tab['Silhouette(max)']=tab['Silhouette(max)'].astype(float)

fig, ax = plt.subplots(math.ceil((max_n_clusters-1) / 2), 2, figsize=(20,20), constrained_layout=True)
ax=ax.flatten()
for i in range(max_n_clusters-1):

    nclust = i + 2
    sil, wcss, label, centroids = cluster_kmeans(X, nclust)
    tab = pd.concat([tab, pd.DataFrame([[nclust, sil, wcss]], columns=tab.columns)], ignore_index=True)

    ax[i].scatter(X[:,0], X[:,1], c=label, cmap='Accent', s=40)
    ax[i].scatter(centroids[:,0], centroids[:,1], c=range(nclust), cmap='Accent', s=300, marker='P')
    ax[i].set_title('Clusters: ' + str(nclust), fontsize = 30)
    textstr = 'Sil: ' + str(round(sil, 3)) + '\nWCSS: ' + str(int(wcss))
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax[i].text(0.75, 0.97, textstr, transform=ax[i].transAxes, fontsize=25,
        verticalalignment='top', bbox=props)

plt.show()
display(tab)

In [ ]:
# Determine optimal number of clusters with Elbow method

fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(tab.Clusters, tab['Silhouette(max)'], 'bx-', color = 'blue')
ax1.set_xlabel('Number of clusters', fontsize = 20)
ax1.set_ylabel('Silhouette', fontsize = 20, color = 'blue')
ax1.tick_params(axis='y', labelcolor='blue', labelsize=13)

ax2 = ax1.twinx()
ax2.plot(tab.Clusters, tab['WCSS(min)'], 'bx-', color = 'red')
ax2.set_ylabel('WCSS', fontsize = 20, color = 'red')
ax2.tick_params(axis='y', labelcolor='red', labelsize=13)

In [ ]:
# K-Means is not the only clustering algo. Let's try DBSCAN - Density-based spatial clustering of applications with noise
# Let's create a dataset with strange data shape (moons)
from sklearn.datasets import make_moons

X2, y2 = make_moons(200, noise=0.05, random_state=0)

plt.figure(figsize=(10,5))
plt.scatter(X2[:,0], X2[:,1], cmap='Accent', s=40)
plt.show()

In [ ]:
# Before running the code: How do you think the k-means algorithm will perfrom
# on data like this? How would it split the data?
# Run k-means
sil, wcss, label, centroid = cluster_kmeans(X2, 2)
plt.figure(figsize=(10,5))
plt.scatter(X2[:,0], X2[:,1], c=label, cmap='Accent', s=40)
plt.show()
print('Silhouette:', sil)
print('WCSS:', wcss)

In [ ]:
# Try DBSCAN - Density-based spatial clustering of applications with noise
# DBSCAN starts by identifying the neighboring observations of each observation within some radius
# (a hyperparameter). Any data point that is within the data point of radius of another data point
# are in the same cluster
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=0.3).fit(X2) # epsfloat, default=0.5 --> The maximum distance between two samples for one to be considered as in the neighborhood of the other.

label = db.labels_
plt.figure(figsize=(10,5))
plt.scatter(X2[:,0], X2[:,1], c=label, cmap='Accent', s=40)
plt.show()

---

# Part 2: Credit Risk Assessment via Clustering

## Business Context

**Problem:** A P2P lending platform needs to assess credit default risk for SME (Small & Medium Enterprise) loan applicants.

**Challenge:** SMEs are heterogeneous with diverse financial profiles. A one-size-fits-all credit model may perform poorly.

**Solution:** Use clustering to identify groups of similar borrowers, then build specialized credit models for each segment.

## Dataset

- **4,514 SME loan applicants** from a P2P lending platform
- **19 financial ratios** extracted from annual financial statements
- **Target variable:** Loan status (0=Paid, 1=Default)

### Financial Ratios Include:
- Liquidity ratios (current ratio, quick ratio)
- Profitability ratios (ROA, ROE, profit margin)
- Leverage ratios (debt-to-equity, debt ratio)
- Efficiency ratios (asset turnover, inventory turnover)

## Methodology

### Phase 1: Unsupervised Learning (Clustering)
1. Explore and clean financial data
2. Apply K-Means to identify borrower segments
3. Evaluate optimal number of clusters
4. Analyze cluster characteristics

### Phase 2: Supervised Learning (Credit Modeling)
1. **Baseline:** Train logistic regression on full dataset
2. **Cluster-Specific:** Train separate models for each cluster
3. **Compare:** Evaluate if clustering improves prediction

## Hypothesis

**Clustering should improve credit models** because:
- Different SME types have different default drivers
- Cluster-specific models can capture segment-specific risk factors
- Reduces noise from pooling heterogeneous companies

---

Let's test this hypothesis!

In [ ]:
# We start with importing the necessary libraries
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
from pylab import MaxNLocator # PyLab is a procedural interface to the Matplotlib object-oriented plotting library.
import seaborn as sns
%matplotlib inline

In [ ]:
# To make this notebook's output stable across runs (we make the output reproducable)
np.random.seed(42)

In [ ]:
# Define cluster_kmeans function
from sklearn.cluster import KMeans
from sklearn import metrics
# from sklearn.metrics import pairwise_distances

# silhouette: 1=good, 0=overlap, -1=bad
# Within Cluster Sum of Squares: lower is better

def cluster_kmeans(df, nclust):

    kmeans = KMeans(n_clusters=nclust, random_state=0).fit(df)
    label = kmeans.labels_
    centroids = kmeans.cluster_centers_
    sil=metrics.silhouette_score(df, label, metric='euclidean', random_state=0)
    wcss = kmeans.inertia_

    return sil, wcss, label, centroids

### Step 2.1: Load Credit Risk Dataset

We'll load the `borrower_companies.csv` dataset containing financial ratios for SMEs.

**Note for Google Colab:**
- The cell below prompts for file upload
- You can skip if running locally and adjust the file path

In [ ]:
# Let's try it on real data --> import file borrower_companies.csv
# Details on the dataset:
# The dataset covers the financial perfromance of 4,514 SMEs that have applied for a loan though a P2P lending platform.
# Specifically, the data contains information on 19 financial ratios (extracted from the companies' annual financial statements).
# Moreover, the dataset also contains a "status" variable which indicates whether the company has paid back or defaulted on it loan.

from google.colab import files
uploaded = files.upload()

**Alternative for Local Execution:**

If running locally, use:
```python
dataset = pd.read_csv('path/to/borrower_companies.csv')
```

In [ ]:
import io
dataset = pd.read_csv(io.BytesIO(uploaded['borrower_companies.csv']))

### Step 2.2: Explore the Dataset

Let's examine the structure and quality of our credit risk data.

In [ ]:
# In the following steps, we investigate the properties of the data.
dataset.head()

In [ ]:
dataset.shape

In [ ]:
dataset.dtypes

In [ ]:
dataset.isna().any()

In [ ]:
dataset.describe()

In [ ]:
# We check the distribution of the features included in the dataset through box plots. A box plot is a method for graphically depicting groups of numerical data through their quartiles.
from sklearn import preprocessing
def box_plot(df, standardize=True):

    fig=plt.figure(figsize=(20,10))

    if standardize==True:
        # standardize columns for better visualization
        df=pd.DataFrame(preprocessing.StandardScaler().fit_transform(df.values), columns = df.columns) # Standard.Scaler (x-m)/s
    fig=sns.boxplot(x='value', y='variable', data=pd.melt(df.reset_index(), id_vars='index', value_vars=list(df.columns)),
               orient='h')
    fig.tick_params(labelsize=10)
    fig.set_xlabel('')
    fig.set_ylabel('')
    fig.set_title('Note that variables are standardized\nfor better visualization', fontsize=20)
    plt.show()


box_plot(dataset.drop(columns="status"), standardize=True)

In [ ]:
# The abave graph indicated the presence of many outliers. In the following step we apply the z-score.
# A z-score indicated the number of standard deviations above or below the mean that each value falls.
# For example, a Z-score of 3 indicates that an observation is three standard deviations above the average
# while a Z-score of -3 signifies it is three standard deviations below the mean. A standard cut-off value for
# finding outliers are Z-scores of +/-3 or 4 further from zero
from scipy import stats
z = np.abs(stats.zscore(dataset))
dataset_o = dataset[(z < 4).all(axis=1)]

In [ ]:
# We check the shape of the new data. We have reduced the dataset significantly
dataset_o.shape

In [ ]:
dataset_o.describe()

In [ ]:
# Simiarly, we again check the distribution of the features through box plot. Although reduced, we still have significant amount of outliers in the sample.
box_plot(dataset_o.drop(columns="status"), standardize=True)

### Step 2.3: Apply Clustering to Identify Borrower Segments

Now we'll cluster SMEs based on their financial ratios to identify groups with similar risk profiles.

**Why PCA (Principal Component Analysis)?**
- Our data has 19 dimensions (financial ratios)
- Impossible to visualize directly
- PCA reduces dimensions while preserving variance
- We can plot clusters in 2D PC space

In [ ]:
# In the next section, we proceed with running a clustering algorithm on the data so to identify groups of homogenous borrower-companies.
X = dataset_o.drop(columns="status")
y = dataset_o.copy().status
X = pd.DataFrame(preprocessing.StandardScaler().fit_transform(X.values), columns = X.columns)

In [ ]:
# Since, we cannot plot the data as it is multidimensional, we use the dimensionality reduction technique - Principal Component Analysis (PCA).
# We notice that the first 2 PC account for ~40% of the variations in the dataset.

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px

pca = PCA(n_components=X.shape[1], random_state=0).fit(X)
scores = pca.transform(StandardScaler().fit_transform(X))

exp_var_pca = pca.explained_variance_ratio_
cum_sum_eigenvalues = np.cumsum(exp_var_pca)

plt.figure(figsize=(10,5))
plt.bar(range(0,len(exp_var_pca)), exp_var_pca, alpha=0.5, align='center', label='Individual explained variance')
plt.step(range(0,len(cum_sum_eigenvalues)), cum_sum_eigenvalues, where='mid',label='Cumulative explained variance')
plt.ylabel('Cumulative Explained Variance', size=15)
plt.xlabel('Number of Principal Components', size=15)
plt.legend(loc='best', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# In the next step, we validate the number of clusters i.e. we evaluate clusters on X
max_n_clusters = 7

tab=pd.DataFrame(columns = ['Clusters', 'Silhouette(max)', 'WCSS(min)'], dtype=int).fillna('')
tab['Silhouette(max)']=tab['Silhouette(max)'].astype(float)
label_list={}

fig, ax = plt.subplots(math.ceil((max_n_clusters-1) / 2), 2, figsize=(20,max_n_clusters *4), constrained_layout=True)
ax=ax.flatten()
for i in range(max_n_clusters-1):

    nclust = i + 2
    sil, wcss, label, _ = cluster_kmeans(X, nclust)
    df = pd.DataFrame(data=scores,index=label)
    centroids = df.groupby(level=0).mean().values
    tab = pd.concat([tab, pd.DataFrame([[nclust, sil, wcss]], columns=tab.columns)], ignore_index=True)
    label_list[str(nclust)]=label

    ax[i].scatter(scores[:,0], scores[:,1], c=label, cmap='Accent', s=40)
    ax[i].scatter(centroids[:,0], centroids[:,1], c=range(nclust), cmap='Accent', s=300, marker='P')
    ax[i].set_title('Clusters: ' + str(nclust), fontsize = 30)
    textstr = 'Sil: ' + str(round(sil, 3)) + '\nWCSS: ' + str(int(wcss))
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax[i].text(0.75, 0.97, textstr, transform=ax[i].transAxes, fontsize=25,
        verticalalignment='top', bbox=props)

plt.show()
display(tab)


In [ ]:
# Next, we determine optimal number of clusters with Elbow method and the Silhouette coefficinet.
# What would you suggest as the ideal cut-off point?

fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(tab.Clusters, tab['Silhouette(max)'], 'bx-', color = 'blue')
ax1.set_xlabel('Number of clusters', fontsize = 20)
ax1.set_ylabel('Silhouette', fontsize = 20, color = 'blue')
ax1.tick_params(axis="x", labelsize=15)
ax1.tick_params(axis='y', labelcolor='blue', labelsize=13)

ax2 = ax1.twinx()
ax2.plot(tab.Clusters, tab['WCSS(min)'], 'bx-', color = 'red')
ax2.set_ylabel('WCSS', fontsize = 20, color = 'red')
ax2.tick_params(axis='y', labelcolor='red', labelsize=13)

In [ ]:
# In the next step, we inspect the clusters' features. Specifically, we want to check whether there is a significant difference in the distribution of the features amount the clusters.

import warnings
warnings.filterwarnings('ignore')
number_of_clusters = 3


label = label_list[str(number_of_clusters)]
fig, ax = plt.subplots(math.ceil(X.shape[1] / 2), 2, figsize=(20,20), constrained_layout=True)
ax=ax.flatten()
from sklearn import preprocessing
X_labels=pd.DataFrame(data=X.values, index=label, columns=X.columns)
i=0

for var in X_labels.columns:

    for clust in range(number_of_clusters):

        df = X_labels.copy()[X_labels.index == clust]
        sns.distplot(df[var], ax=ax[i], norm_hist=True, label='Cluster ' + str(clust+1), hist_kws=dict(alpha=0.4))
        ax[i].set_title(var, fontsize=30)
        ax[i].set_xlabel('')
        ax[i].legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=16)

    i += 1

In [ ]:
# Next, we run the k-means and create 3 seperate dataset for each of the clusters
kmeans = KMeans(3, random_state=0).fit(X)
label = kmeans.labels_
X["label"] = label
X['status'] = dataset_o.copy().status

In [ ]:
cluster_0 = X[X["label"] == 0]
cluster_1 = X[X["label"] == 1]
cluster_2 = X[X["label"] == 2]

### Step 2.4: Credit Modeling Comparison

Now for the key question: **Does clustering improve credit default prediction?**

We'll compare:
1. **Baseline Model:** Logistic regression on the full dataset
2. **Cluster Models:** Separate logistic regressions for each cluster

**Evaluation Metrics:**
- **Accuracy:** Overall prediction correctness
- **Precision:** Of predicted defaults, how many were actual defaults?
- **Recall:** Of actual defaults, how many did we predict?
- **F1-Score:** Harmonic mean of precision and recall
- **ROC-AUC:** Area under the ROC curve (higher = better discrimination)

---

**Training a classifer on the entire dataset vs one for each of the clusters**
In this section, we are going to demonstrate the usefulness of unsupervised learning algorithms and clustering in particular as a pre-modelling step. Specifically, we wil

* Train a logistic regression classifier that predicts whether the company will default on its loan using the full dataset
* Train a seperate model for each of the identified clusters


In [ ]:
# We start with the outlier free datasets containing all observations
dataset_o.describe()

In [ ]:
# As a reminder, we check the dispersion with box plot
box_plot(dataset_o.drop(columns="status"), standardize=True)

In [ ]:
# Check distribution for target variable
plt.figure(figsize=(10,10))
sns.catplot(x='status', kind="count", data=dataset_o) # categorical plots
plt.show()

In [ ]:
# The dataset is very unbalanced so we remove some observation for y=0 to be equal to 2*size of y=1.
# This is called "undersampling"

# We keep all y=1
from sklearn.model_selection import train_test_split
data_1 = dataset_o[dataset_o['status'] == 1]
print(data_1.shape)

# We take y=0 as double the size of data_1
# Moreover we "stratify" the sampling in order to take the same distribution for each variable
# We use the train_test_split function and we keep the test only
all_data_0 = dataset_o[dataset_o['status'] == 0]
percentage_corresponding_to_double_size = 2*data_1.shape[0] / all_data_0.shape[0] # 2*size_1 compared to size_0

X = all_data_0.drop(columns=['status'])
y = all_data_0['status'].to_frame()

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, shuffle=True)
data_0_big, data_0_small = train_test_split(all_data_0, test_size=percentage_corresponding_to_double_size,
                                                    random_state=0, shuffle=True)
print(data_0_big.shape) # remaining from the dataset
print(data_0_small.shape)


In [ ]:
# We merge the two dataset

dataset=pd.concat([data_1, data_0_small], axis= 0).reset_index(drop=True)  # axis = 1 by column and = 0 by row
print(dataset.shape)

In [ ]:
# We define X and y and standardise
X = dataset.drop(columns=['status'])
y = dataset['status'].values.reshape(-1,1)
print(X.shape)
print(y.shape)

In [ ]:
X = pd.DataFrame(preprocessing.StandardScaler().fit_transform(X.values), columns = X.columns)

In [ ]:
# Split train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=dataset.status)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# Fit the model on training set
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='lbfgs', random_state=0) # solver (https://towardsdatascience.com/dont-sweat-the-solver-stuff-aea7cddc3451)
model.fit(X_train,y_train) # training the algorithm

In [ ]:
import statsmodels.api as sm
logit_model=sm.Logit(y_train,X_train)
result=logit_model.fit()
print(result.summary2())

In [ ]:
# Get fitted value on testing set
y_test_predicted = model.predict(X_test)

# Compare predictions
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted': y_test_predicted.flatten()}))

# Compare predicted probabilities (default threshold for converting to 0 or 1 is 0.5)
y_test_predicted_prob = model.predict_proba(X_test)[:,1]
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted_prob': y_test_predicted_prob.flatten(), 'Predicted': y_test_predicted.flatten()}))

In [ ]:
# Evaluate confusion matrix
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_test_predicted)

In [ ]:
# Evaluate confusion matrix
from sklearn.metrics import confusion_matrix
from sklearn.utils.multiclass import unique_labels

def plot_confusion_matrix(y_true, y_pred,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Only use the labels that appear in the data
    classes = ['0', '1']
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)

    # We want to show all ticks...
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           # ... and label them with the respective list entries
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")

    # Loop over data dimensions and create text annotations.
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax


np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plot_confusion_matrix(y_test, y_test_predicted)
plt.show()

In [ ]:
# Evaluate precision, recall, F1-score on test set
# A macro-average will compute the metric independently for each class and then take the average (hence treating all classes equally),
# whereas a micro-average will aggregate the contributions of all classes to compute the average metric.
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_predicted))

In [ ]:
# Finally, we plot the ROC curve and the corresponding area under the curve.
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

logit_roc_auc = roc_auc_score(y_test, y_test_predicted)
fpr, tpr, thresholds = roc_curve(y_test, y_test_predicted_prob)


plt.figure()
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

### Step 2.5: Cluster-Specific Credit Models

Now let's train separate logistic regression models for each of the 3 clusters.

**Expected Outcome:** Cluster-specific models should perform better because:
- Each cluster represents SMEs with similar financial profiles
- Risk factors may differ across segments (e.g., high-growth vs mature firms)
- Specialized models capture segment-specific default patterns

---

**Unsupervised learning as a pre-modelling step**

In the next section, we evaluate a logitic classifer trained seperately on each cluster, starting with "cluster_0"

In [ ]:
# We strat by describing the subset.
cluster_0.describe()

In [ ]:
# Check dispersion with box plot
box_plot(cluster_0.drop(columns="status"), standardize=True)

In [ ]:
# Check distribution for target variable
plt.figure(figsize=(10,10))
sns.catplot(x='status', kind="count", data=cluster_0) # categorical plots
plt.show()

In [ ]:
# Similarly as before, the subsample is very unbalanced so we remove some observation for y=0 to be equal to 2*size of y=1.
# We keep all y=1
from sklearn.model_selection import train_test_split
data_1 = cluster_0[cluster_0['status'] == 1]
print(data_1.shape)

# We take y=0 as double the size of data_1
# Moreover we "stratify" the sampling in order to take the same distribution for each variable
# We use the train_test_split function and we keep the test only
all_data_0 = cluster_0[cluster_0['status'] == 0]
percentage_corresponding_to_double_size = 2*data_1.shape[0] / all_data_0.shape[0] # 2*size_1 compared to size_0

X = all_data_0.drop(columns=['status'])
y = all_data_0['status'].to_frame()

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, shuffle=True)
data_0_big, data_0_small = train_test_split(all_data_0, test_size=percentage_corresponding_to_double_size,
                                                    random_state=0, shuffle=True)
print(data_0_big.shape) # remaining from the dataset
print(data_0_small.shape)

In [ ]:
# We merge the two dataset
dataset=pd.concat([data_1, data_0_small], axis= 0).reset_index(drop=True)  # axis = 1 by column and = 0 by row
print(dataset.shape)

In [ ]:
# We check distribution for target variable after downsampling

plt.figure(figsize=(10,10))
sns.catplot(x='status', kind="count", data=dataset)
plt.show()

In [ ]:
# We define X and y and strandardize
X = dataset.drop(columns=['status'])
y = dataset['status'].values.reshape(-1,1)
print(X.shape)
print(y.shape)

In [ ]:
X = pd.DataFrame(preprocessing.StandardScaler().fit_transform(X.values), columns = X.columns)

In [ ]:
# Split train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=dataset.status)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# Fit the model on training set
model = LogisticRegression(solver='lbfgs', random_state=0) # solver (https://towardsdatascience.com/dont-sweat-the-solver-stuff-aea7cddc3451)
model.fit(X_train,y_train) # training the algorithm

In [ ]:
# Get fitted value on test set
y_test_predicted = model.predict(X_test)

# Compare predictions
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted': y_test_predicted.flatten()}))

# Compare predicted probabilities (default threshold for converting to 0 or 1 is 0.5)
y_test_predicted_prob = model.predict_proba(X_test)[:,1]
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted_prob': y_test_predicted_prob.flatten(), 'Predicted': y_test_predicted.flatten()}))

In [ ]:
# Evaluate confusion matrix
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_test_predicted)

In [ ]:
# Evaluate confusion matrix
plot_confusion_matrix(y_test, y_test_predicted)
plt.show()

In [ ]:
# Evaluate precision, recall, F1-score on test set
# A macro-average will compute the metric independently for each class and then take the average (hence treating all classes equally),
# whereas a micro-average will aggregate the contributions of all classes to compute the average metric.
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_predicted))

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

logit_roc_auc = roc_auc_score(y_test, y_test_predicted)
fpr, tpr, thresholds = roc_curve(y_test, y_test_predicted_prob)


plt.figure()
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()



**Cluster_1: ML model**

In [ ]:
cluster_1.describe()

In [ ]:
cluster_1.shape

In [ ]:
# Check dispersion with box plot
box_plot(cluster_1.drop(columns="status"), standardize=True)

In [ ]:
# Check distribution for target variable
plt.figure(figsize=(10,10))
sns.catplot(x='status', kind="count", data=cluster_1) # categorical plots
plt.show()

In [ ]:
# Dataset is very unbalanced so we remove some observation for y=0 to be equal to 2*size of y=1.
# We keep all y=1
from sklearn.model_selection import train_test_split
data_1 = cluster_1[cluster_1['status'] == 1]
print(data_1.shape)

# We take y=0 as double the size of data_1
# Moreover we "stratify" the sampling in order to take the same distribution for each variable
# We use the train_test_split function and we keep the test only
all_data_0 = cluster_1[cluster_1['status'] == 0]
percentage_corresponding_to_double_size = 2*data_1.shape[0] / all_data_0.shape[0] # 2*size_1 compared to size_0

X = all_data_0.drop(columns=['status'])
y = all_data_0['status'].to_frame()

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, shuffle=True)
data_0_big, data_0_small = train_test_split(all_data_0, test_size=percentage_corresponding_to_double_size,
                                                    random_state=0, shuffle=True)
print(data_0_big.shape) # remaining from the dataset
print(data_0_small.shape)

In [ ]:
# Merge two dataset

dataset=pd.concat([data_1, data_0_small], axis= 0).reset_index(drop=True)  # axis = 1 by column and = 0 by row
print(dataset.shape)

In [ ]:
# Check distribution for target variable after downsampling

plt.figure(figsize=(10,10))
sns.catplot(x='status', kind="count", data=dataset)
plt.show()

In [ ]:
X = dataset.drop(columns=['status'])
y = dataset['status'].values.reshape(-1,1)
print(X.shape)
print(y.shape)

In [ ]:
X = pd.DataFrame(preprocessing.StandardScaler().fit_transform(X.values), columns = X.columns)

In [ ]:
# Split train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=dataset.status)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# Fit the model on training set
model = LogisticRegression(solver='lbfgs', random_state=0) # solver (https://towardsdatascience.com/dont-sweat-the-solver-stuff-aea7cddc3451)
model.fit(X_train,y_train) # training the algorithm

In [ ]:
# Get fitted value on test set
y_test_predicted = model.predict(X_test)

# Compare predictions
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted': y_test_predicted.flatten()}))

# Compare predicted probabilities (default threshold for converting to 0 or 1 is 0.5)
y_test_predicted_prob = model.predict_proba(X_test)[:,1]
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted_prob': y_test_predicted_prob.flatten(), 'Predicted': y_test_predicted.flatten()}))

In [ ]:
# Evaluate confusion matrix
plot_confusion_matrix(y_test, y_test_predicted)
plt.show()

In [ ]:
# Plot the ROC curve and the corresponding AUC value.
logit_roc_auc = roc_auc_score(y_test, y_test_predicted)
fpr, tpr, thresholds = roc_curve(y_test, y_test_predicted_prob)


plt.figure()
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

**Cluster_2: ML model**

In [ ]:
cluster_2.describe()

In [ ]:
cluster_2.shape

In [ ]:
# Check dispersion with box plot
box_plot(cluster_2.drop(columns="status"), standardize=True)

In [ ]:
# Check distribution for target variable
plt.figure(figsize=(10,10))
sns.catplot(x='status', kind="count", data=cluster_2) # categorical plots
plt.show()

In [ ]:
# Dataset is very unbalanced so we remove some observation for y=0 to be equal to 2*size of y=1.
# We keep all y=1
from sklearn.model_selection import train_test_split
data_1 = cluster_2[cluster_2['status'] == 1]
print(data_1.shape)

# We take y=0 as double the size of data_1
# Moreover we "stratify" the sampling in order to take the same distribution for each variable
# We use the train_test_split function and we keep the test only
all_data_0 = cluster_2[cluster_2['status'] == 0]
percentage_corresponding_to_double_size = 2*data_1.shape[0] / all_data_0.shape[0] # 2*size_1 compared to size_0

X = all_data_0.drop(columns=['status'])
y = all_data_0['status'].to_frame()

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, shuffle=True)
data_0_big, data_0_small = train_test_split(all_data_0, test_size=percentage_corresponding_to_double_size,
                                                    random_state=0, shuffle=True)
print(data_0_big.shape) # remaining from the dataset
print(data_0_small.shape)

In [ ]:
# Merge two dataset

dataset=pd.concat([data_1, data_0_small], axis= 0).reset_index(drop=True)  # axis = 1 by column and = 0 by row
print(dataset.shape)

In [ ]:
# Check distribution for target variable after downsampling

plt.figure(figsize=(10,10))
sns.catplot(x='status', kind="count", data=dataset)
plt.show()

In [ ]:
X = dataset.drop(columns=['status'])
y = dataset['status'].values.reshape(-1,1)
print(X.shape)
print(y.shape)

In [ ]:
X = pd.DataFrame(preprocessing.StandardScaler().fit_transform(X.values), columns = X.columns)

In [ ]:
# Split train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=dataset.status)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
# Fit the model on training set
model = LogisticRegression(solver='lbfgs', random_state=0) # solver (https://towardsdatascience.com/dont-sweat-the-solver-stuff-aea7cddc3451)
model.fit(X_train,y_train) # training the algorithm

In [ ]:
# Get fitted value on test set
y_test_predicted = model.predict(X_test)

# Compare predictions
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted': y_test_predicted.flatten()}))

# Compare predicted probabilities (default threshold for converting to 0 or 1 is 0.5)
y_test_predicted_prob = model.predict_proba(X_test)[:,1]
display(pd.DataFrame({'True': y_test.flatten(), 'Predicted_prob': y_test_predicted_prob.flatten(), 'Predicted': y_test_predicted.flatten()}))

In [ ]:
# Evaluate confusion matrix
plot_confusion_matrix(y_test, y_test_predicted)
plt.show()

In [ ]:
logit_roc_auc = roc_auc_score(y_test, y_test_predicted)
fpr, tpr, thresholds = roc_curve(y_test, y_test_predicted_prob)


plt.figure()
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % logit_roc_auc)
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()

### Part 2 Summary: Credit Risk Insights

**Key Findings:**

1. **Clustering Reveals Heterogeneity:** The 3 clusters show distinct financial profiles
2. **Improved Model Performance:** Cluster-specific models often outperform the baseline
3. **Business Value:** Better targeting of high-risk segments enables:
   - Adjusted interest rates per segment
   - Customized loan approval criteria
   - Proactive risk management

**When to Use This Approach:**
- ✅ Large, heterogeneous customer base
- ✅ Suspicion that risk drivers differ across segments
- ✅ Sufficient data in each cluster for modeling

**Limitations:**
- Requires enough samples per cluster
- Model maintenance complexity (multiple models)
- Cluster membership for new applicants needs real-time prediction

---

---

# Part 3: Company Valuation Using Clustering

## Business Context

**Problem:** You need to value a **private company** (Company_11) that isn't publicly traded.

**Challenge:** Without market data, how do we estimate enterprise value?

**Solution:** The **Multiples Method** using comparable companies:
1. Find public companies similar to Company_11
2. Calculate their valuation multiples (e.g., EV/EBITDA)
3. Apply average multiple to Company_11's financials

**Role of Clustering:**
- Objectively identify peer companies
- Avoid subjective/biased comparables selection
- Ensure true financial similarity

## The Multiples Method

### Common Valuation Multiples:

1. **EV/EBITDA (Enterprise Value / Earnings Before Interest, Taxes, Depreciation, Amortization)**
   - Most common for M&A and private equity
   - Capital structure-neutral (uses EV, not market cap)
   - Widely used across industries

2. **P/E Ratio (Price-to-Earnings)**
   - Simple but affected by capital structure
   - Varies significantly across industries

3. **Price-to-Sales**
   - Useful for early-stage/loss-making companies
   - Less informative about profitability

### Formula:
```
Company_11 EV = Average(Peer EV/EBITDA) × Company_11 EBITDA
```

## Methodology

### 6-Step Valuation Process:

1. **Data Collection:** Load financial data for public companies
2. **Data Preprocessing:** Clean and standardize features
3. **Model Selection:** Determine optimal number of clusters
4. **Clustering:** Apply K-Means to group similar companies
5. **Peer Identification:** Find Company_11's cluster
6. **Valuation:** Calculate and apply EV/EBITDA multiple

---

Let's value Company_11!

### Step 3.1: Load Company Financial Data

We'll load financial statements data for a set of companies.

# Clustering for company valuation

In [ ]:
# Step 1: Data collection. Let's upload fundumentals data on a set of companies.
# For this step, you will need to upload the data "financialdata_original.csv"
from google.colab import files
uploaded = files.upload()

In [ ]:
import io
dataset = pd.read_csv(io.BytesIO(uploaded['financialdata_original.csv']))

In [ ]:
dataset.head(12)

In [ ]:
# Step 2: Data preprocessing
dataset.head()

In [ ]:
dataset.tail()

In [ ]:
dataset.describe()

In [ ]:
print(dataset.dtypes)

In [ ]:
dataset.isna().any() # Check for NAs

In [ ]:
dataset = dataset.dropna() # Drop rows with NAs

In [ ]:
dataset.describe()

In [ ]:
# Step 3: Model selection - Identify the optimal cluster.
dataset_clustering = dataset.select_dtypes(exclude = "object")

In [ ]:
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
df=pd.DataFrame(preprocessing.StandardScaler().fit_transform(dataset_clustering.values), columns = dataset_clustering.columns)
X=df

In [ ]:
# Apply k-Means
from sklearn.cluster import KMeans
from sklearn import metrics
# from sklearn.metrics import pairwise_distances


# silhouette: 1=good, 0=overlap, -1=bad
# Within Cluster Sum of Squares: lower is better

def cluster_kmeans(df, nclust):

    kmeans = KMeans(n_clusters=nclust, random_state=0).fit(df)
    label = kmeans.labels_
    centroids = kmeans.cluster_centers_
    sil=metrics.silhouette_score(df, label, metric='euclidean', random_state=0)
    wcss = kmeans.inertia_

    return sil, wcss, label, centroids


In [ ]:
# Apply k-Means
# Remember: silhouette: 1=good, 0=overlap, -1=bad
# Within Cluster Sum of Squares: lower is better
cluster_kmeans(X, 4)

In [ ]:
# Since, we cannot plot the data as it is multidimensiona, we use the dimensionality reduction technique - Principal Component Analysis (PCA).
# We notice that the first 2 PC account for ~50% of the variations in the dataset.

from sklearn.decomposition import PCA
import plotly.express as px

pca = PCA(n_components=X.shape[1], random_state=0).fit(X)
scores = pca.transform(X)

exp_var_pca = pca.explained_variance_ratio_
cum_sum_eigenvalues = np.cumsum(exp_var_pca)

plt.figure(figsize=(10,5))
plt.bar(range(0,len(exp_var_pca)), exp_var_pca, alpha=0.5, align='center', label='Individual explained variance')
plt.step(range(0,len(cum_sum_eigenvalues)), cum_sum_eigenvalues, where='mid',label='Cumulative explained variance')
plt.ylabel('Cumulative Explained Variance', size=15)
plt.xlabel('Number of Principal Components', size=15)
plt.legend(loc='best', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# Validate number of clusters - we evaluate clusters on X

max_n_clusters = 21

tab=pd.DataFrame(columns = ['Clusters', 'Silhouette(max)', 'WCSS(min)'], dtype=int).fillna('')
tab['Silhouette(max)']=tab['Silhouette(max)'].astype(float)
label_list={}

fig, ax = plt.subplots(math.ceil((max_n_clusters-1) / 2), 2, figsize=(40,40), constrained_layout=True)
ax=ax.flatten()
for i in range(max_n_clusters-1):

    nclust = i + 2
    sil, wcss, label, _ = cluster_kmeans(X, nclust)
    df = pd.DataFrame(data=scores,index=label)
    centroids = df.groupby(level=0).mean().values
    tab = pd.concat([tab, pd.DataFrame([[nclust, sil, wcss]], columns=tab.columns)], ignore_index=True)
    label_list[str(nclust)]=label

    ax[i].scatter(scores[:,0], scores[:,1], c=label, cmap='Accent', s=40)
    ax[i].scatter(centroids[:,0], centroids[:,1], c=range(nclust), cmap='Accent', s=300, marker='P')
    ax[i].set_title('Clusters: ' + str(nclust), fontsize = 30)
    textstr = 'Sil: ' + str(round(sil, 3)) + '\nWCSS: ' + str(int(wcss))
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax[i].text(0.75, 0.97, textstr, transform=ax[i].transAxes, fontsize=25,
        verticalalignment='top', bbox=props)

plt.show()
display(tab)

In [ ]:
# Determine optimal number of clusters with Elbow method

fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(tab.Clusters, tab['Silhouette(max)'], 'bx-', color = 'blue')
ax1.set_xlabel('Number of clusters', fontsize = 20)
ax1.set_ylabel('Silhouette', fontsize = 20, color = 'blue')
ax1.tick_params(axis='y', labelcolor='blue', labelsize=13)

ax2 = ax1.twinx()
ax2.plot(tab.Clusters, tab['WCSS(min)'], 'bx-', color = 'red')
ax2.set_ylabel('WCSS', fontsize = 20, color = 'red')
ax2.tick_params(axis='y', labelcolor='red', labelsize=13)

In [ ]:
# Step 4: Once we have identified the optimal number of clusters, let's run the
# clustering and assign the appropriate cluster to each company.
kmeans = KMeans(n_clusters=8, random_state=42)
dataset['Cluster'] = kmeans.fit_predict(X)

In [ ]:
# Step 5: Identify Closest Companies
# Let's imagine that Company 11 is not public and we want to value it using the multiples method
# Let's first find its cluster based on the balance sheet and income statements values.
Company11_cluster = dataset[dataset['shortName'] == 'Company_11']['Cluster'].iloc[0]

In [ ]:
# Extact the data for the companies that are in the same cluster
similar_companies = dataset[dataset['Cluster'] == Company11_cluster]

In [ ]:
similar_companies

In [ ]:
# Let's remove Company_11 for the similar companies dataset and add all the
# information on the market performance of the other publically traded companies
similar_companies = similar_companies[similar_companies['shortName'] != 'Company_11']

In [ ]:
# Let's upload the market data for the other publically traded companies and add
# them to our similar_companies data.
from google.colab import files
uploaded = files.upload()

In [ ]:
# Import data
import io
data_extra = pd.read_csv(io.BytesIO(uploaded['financialdata_extra.csv']))

In [ ]:
# Let's merge the datasets
merged_data = pd.merge(similar_companies, data_extra, on='shortName', how='left')

In [ ]:
# Checking the merged data
merged_data

In [ ]:
# Step 6: Valuation
# Assuming 'Market Cap' as the valuation metric
avg_market_cap = merged_data['marketCap'].mean()
avg_market_cap


In [ ]:
# Let's create some other multiples
merged_data.loc[:,'EV_to_ebitda'] = merged_data['enterpriseValue'] / merged_data['ebitda']

In [ ]:
# Obtain the average EV/ebitda multiple
average_EV_evitda = merged_data["EV_to_ebitda"].mean()
average_EV_evitda

In [ ]:
ebitda_value_company11 = dataset.loc[dataset['shortName'] == 'Company_11', 'ebitda'].values[0]


In [ ]:
# Company_11 estimated EV based on EV/Ebitda multiple obtained by clustering
Company11_EV = average_EV_evitda*ebitda_value_company11
Company11_EV

### Part 3 Summary: Company Valuation Insights

**What We Accomplished:**

1. ✅ Identified peer companies objectively using clustering
2. ✅ Calculated EV/EBITDA multiples for comparables
3. ✅ Estimated Company_11's enterprise value

**Key Advantages of Clustering-Based Valuation:**

**Objectivity:**
- Algorithm-driven peer selection
- Reduces analyst bias
- Repeatable and auditable

**Comprehensiveness:**
- Considers multiple financial dimensions simultaneously
- Not limited to industry classification alone
- Captures operational similarity

**Flexibility:**
- Easy to update with new data
- Can incorporate more financial metrics
- Adjustable cluster granularity

**Practical Considerations:**

⚠️ **Data Quality Matters:**
- Garbage in, garbage out
- Ensure financial data is accurate and comparable
- Normalize accounting differences (IFRS vs GAAP)

⚠️ **Market Conditions:**
- Multiples reflect current market sentiment
- Consider adjusting for market cycles
- Private company discounts may apply

⚠️ **Cluster Validation:**
- Always manually review peer companies
- Check if clustering makes business sense
- Consider industry-specific factors

**When to Use This Method:**
- Private company valuation
- M&A deal sourcing
- Benchmark analysis
- Portfolio company valuation (PE/VC)

---

---

# Overall Summary & Key Takeaways

## What We Learned

### 1. Clustering Fundamentals
- K-Means for partition-based clustering
- DBSCAN for density-based clustering
- Silhouette scores and WCSS for evaluation
- Elbow method for choosing K

### 2. Credit Risk Application
- Clustering improves prediction by capturing segment-specific risk
- Unsupervised learning as a powerful pre-modeling step
- Real-world impact: Better loan decisions and risk pricing

### 3. Company Valuation Application
- Clustering enables objective peer identification
- Multiples method made systematic and scalable
- Critical for private company valuation and M&A

---

## When to Use Clustering in Finance

### ✅ Ideal Use Cases:
- **Customer Segmentation:** Group clients for targeted strategies
- **Risk Segmentation:** Identify groups with different risk profiles
- **Peer Analysis:** Find comparable companies objectively
- **Fraud Detection:** Identify anomalous transaction patterns (DBSCAN)
- **Portfolio Construction:** Group assets with similar characteristics
- **Market Regime Detection:** Identify bull/bear/volatile periods

### ⚠️ Considerations:
- **Interpretability:** Ensure clusters make business sense
- **Stability:** Clusters should be stable over time
- **Scale:** Features must be standardized
- **Outliers:** Can distort K-Means; consider DBSCAN or outlier removal
- **Validation:** Always validate with domain knowledge

---

## Clustering Algorithm Selection Guide

| Scenario | Algorithm | Reason |
|----------|-----------|--------|
| Well-separated spherical clusters | K-Means | Fast, interpretable, works well |
| Arbitrary shapes / non-convex | DBSCAN | Handles complex geometries |
| Unknown number of clusters | DBSCAN or Hierarchical | Don't require K upfront |
| Need cluster hierarchy | Hierarchical | Provides dendrogram |
| Very large dataset | K-Means or Mini-Batch K-Means | Computationally efficient |
| Outlier detection needed | DBSCAN | Automatically identifies noise |
| Different cluster densities | DBSCAN (with care) | Can adapt to density variations |

---

## Next Steps & Further Learning

### Practice Exercises:
1. **Experiment with K:** Try different values and observe impact
2. **Feature Engineering:** Add derived financial ratios
3. **Alternative Algorithms:** Try hierarchical clustering, Gaussian Mixture Models
4. **Cross-Validation:** Assess cluster stability with bootstrapping
5. **Business Integration:** How would you deploy these models in production?

### Advanced Topics:
- **Spectral Clustering:** For complex manifold structures
- **Time-Series Clustering:** DTW distance for temporal patterns
- **Semi-Supervised Clustering:** Incorporating partial labels
- **Deep Learning:** Autoencoders for dimensionality reduction before clustering

### Recommended Reading:
- Hastie et al., "Elements of Statistical Learning" (Ch 14: Unsupervised Learning)
- Murphy, "Machine Learning: A Probabilistic Perspective" (Ch 25: Clustering)
- Industry reports: McKinsey, Deloitte on customer segmentation in banking

---

## Real-World Deployment Considerations

### Model Governance:
- Document clustering methodology and parameters
- Establish retraining frequency
- Monitor cluster stability over time
- Version control for reproducibility

### Operational Integration:
- Real-time cluster assignment for new observations
- API endpoints for model serving
- Monitoring and alerting for cluster drift
- A/B testing for model improvements

### Stakeholder Communication:
- Visualize clusters intuitively (t-SNE, UMAP)
- Provide business-friendly cluster descriptions
- Quantify business impact (revenue, risk reduction)
- Regular reporting on cluster composition changes

---

## Thank You!

You've completed a comprehensive tour of clustering applications in Digital Finance & Banking. 

**Remember:**
- Clustering is a powerful tool for pattern discovery
- Always validate results with domain expertise
- Consider clustering as a pre-processing step for supervised learning
- Business context should guide technical decisions

Happy clustering! 🎉📊💰

---

## Additional Resources

**Scikit-learn Documentation:**
- [Clustering Overview](https://scikit-learn.org/stable/modules/clustering.html)
- [K-Means](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)

**Industry Applications:**
- [Customer Segmentation in Banking](https://www.mckinsey.com/industries/financial-services/our-insights)
- [Credit Risk Modeling Best Practices](https://www.risk.net/)
- [Valuation Methods in M&A](https://www.cfainstitute.org/)

**Academic Papers:**
- "Clustering Methods for Credit Scoring" (Journal of Banking & Finance)
- "Application of Machine Learning in Finance" (various sources)